In [1]:
import sys
sys.path.append('../../Simulate/')

In [2]:
import os
import random
import numpy as np
import subprocess
import multiprocessing
import threading

from Bio import SeqIO
from tqdm import tqdm
from scipy.stats import bernoulli
from typing import Dict, Union, Tuple
from threading import Lock
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor, as_completed
from multiprocessing import Queue 

In [3]:
working_path = "/home/wbguo/iproject/BSReadSim/test/"
ref_fasta = working_path + 'data/ref/BSB_test.fa'
outdir = working_path + "outdir/"
prefix = 'sim'

# out_queue: thread (1+n)

In [4]:
import importlib
import BSReadSim_queue
importlib.reload(BSReadSim_queue)
import DataProcessor2
importlib.reload(DataProcessor2)

from BSReadSim_queue import BSReadSim
from DataProcessor2 import DataProcessor
from LockedIterator import LockedIterator
from SetMethylation import SetMethylation
from StreamReads import StreamReads
from StreamHTSIM import StreamHTSIM
from SetExperiment import SetExperiment
from ReadProcessor import ReadProcessor

In [5]:
self = BSReadSim(ref_fasta = ref_fasta, outdir=outdir, prefix=prefix, overwrite_db=True, num_reads=10**5,
                 gzip=False, shuffle=False)

Initiating experiment...
Initiating methylation profile...

[Initiating meth_db] for chr10...
Filling with beta distribution for chr10...
Processed 187408 sites from contig chr10

[Initiating meth_db] for chr11...
Filling with beta distribution for chr11...
Processed 187844 sites from contig chr11

[Initiating meth_db] for chr12...
Filling with beta distribution for chr12...
Processed 184249 sites from contig chr12

[Initiating meth_db] for chr13...
Filling with beta distribution for chr13...
Processed 142075 sites from contig chr13

[Initiating meth_db] for chr14...
Filling with beta distribution for chr14...
Processed 154158 sites from contig chr14

[Initiating meth_db] for chr15...
Filling with beta distribution for chr15...
Processed 2252 sites from contig chr15


../../Simulate/StreamReads.py:42: UserWarning: Fastq file exists, will overwrite... /home/wbguo/iproject/BSReadSim/test/outdir//sim_1.fastq
  warnings.warn(f'Fastq file exists, will overwrite... {fastq_file}')
../../Simulate/StreamReads.py:42: UserWarning: Fastq file exists, will overwrite... /home/wbguo/iproject/BSReadSim/test/outdir//sim_2.fastq
  warnings.warn(f'Fastq file exists, will overwrite... {fastq_file}')


In [6]:
self.__dict__

{'ref_fasta': '/home/wbguo/iproject/BSReadSim/test/data/ref/BSB_test.fa',
 'outdir': '/home/wbguo/iproject/BSReadSim/test/outdir/',
 'prefix': 'sim',
 'pair_end': True,
 'n_threads': 4,
 'verbose': True,
 'ref_dict': {'chr10': SeqRecord(seq=Seq('ACTTGCCACTAGAGGAAACCTCACTGATAAAGAGTTAACTGTGTCATTTACCTG...GCT'), id='chr10', name='chr10', description='chr10', dbxrefs=[]),
  'chr11': SeqRecord(seq=Seq('ACTGGAAGGAGAATGGTTAGGACGGGTCCTAGGAGTGGGAGTAACCAGGATTAC...CCC'), id='chr11', name='chr11', description='chr11', dbxrefs=[]),
  'chr12': SeqRecord(seq=Seq('AACCCACTTGACCATGGTGTTTTTTCTTTTCTATATGCTCTGGAATTGGTTTCT...GTA'), id='chr12', name='chr12', description='chr12', dbxrefs=[]),
  'chr13': SeqRecord(seq=Seq('AGTGTGAGAATGGAAGGCTCTTCTTCAAACTATGCAAAATGAATCAATCAAAAG...GGC'), id='chr13', name='chr13', description='chr13', dbxrefs=[]),
  'chr14': SeqRecord(seq=Seq('GTTGTGTGCCACCAGGCCTGGCTAATTTTTGTATTTTTACTTGAGATGGGCATG...GTC'), id='chr14', name='chr14', description='chr14', dbxrefs=[]),
  'chr15': Seq

In [7]:
cmd_part = [self.htsim_path, self.ref_fasta] + [str(item) for key_val in self.htsim_opts.items() for item in key_val]

contig_id = 'chr10'

sim_cmd  = cmd_part + ['-c', contig_id] + ['-n', str(self.count_dict[contig_id])]

read_gen = iter(StreamHTSIM(sim_cmd=sim_cmd, pair_end=self.pair_end)) # only output 1 header for -c TODO:
var_contig, sim_data= next(read_gen)                                    # the first element of generator is variants
self.curr_contig= var_contig                                            # update the profiles
self.var_profile= self.meth_set.set_var_meth(var_contig, sim_data)      # a dict, can be empty
pos_map, meth_arr, _  = self.meth_db.load_contig(var_contig)                              # [pos_map, meth_arr, status]
# self.meth_db.load_contig_share(var_contig)                              # [pos_map, meth_arr, status]

Simulating whole genome reads:
Reference genome file: /home/wbguo/iproject/BSReadSim/test/data/ref/BSB_test.fa
[main] Calculating the total length and effective length of the reference sequences...
[main] Contig chr10 specified, contig length: 423500, effective length: 423500
[main] No VCF input, will generate SNP randomly if mutation rate is nonzero
[htsim] seed = 1679077091
[sim_core] contig 'chr10': simulate 21589 reads...


In [8]:
self.processor  = ReadProcessor(meth_arr= meth_arr,
                                pos_map = pos_map,
                                var_profile = self.var_profile,
                                experiment  = self.experiment)

In [9]:
self.data_processor = DataProcessor(read_gen = read_gen, n_workers=8, 
                                    processor=self.processor, fastq_out=self.fastq_out)

In [10]:
import cProfile
cProfile.run('self.data_processor.start_processing_2q()')

[sim_core] Generated 21589 read pairs, with 2254 contain SNP, 470 contain INDEL


         6482575 function calls in 213.085 seconds

   Ordered by: standard name

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
        1    0.000    0.000  213.085  213.085 <string>:1(<module>)
        1    0.063    0.063  213.084  213.084 DataProcessor2.py:18(start_processing_2q)
    43178    1.134    0.000   26.085    0.001 StreamHTSIM.py:103(process_read_lines)
    21590    0.027    0.000   26.459    0.001 StreamHTSIM.py:19(__iter__)
    21591    0.125    0.000   26.432    0.001 StreamHTSIM.py:53(collect_reads)
    21590    0.031    0.000    0.223    0.000 StreamHTSIM.py:69(get_line)
    21589    0.078    0.000    0.241    0.000 _base.py:316(__init__)
        1    0.000    0.000    0.000    0.000 _base.py:632(__enter__)
        1    0.000    0.000    0.000    0.000 _base.py:635(__exit__)
    43178    0.040    0.000    0.556    0.000 _methods.py:54(_any)
   129534    0.385    0.000    0.415    0.000 _ufunc_config.py:131(geterr)
   129534    0.429    0.000  

# out_queue: thread (1+n) with chunk

In [4]:
import importlib
import BSReadSim_queue
importlib.reload(BSReadSim_queue)
import DataProcessor4
importlib.reload(DataProcessor4)

from BSReadSim_queue import BSReadSim
from DataProcessor4 import DataProcessor
from LockedIterator import LockedIterator
from SetMethylation import SetMethylation
from StreamReads import StreamReads
from StreamHTSIM import StreamHTSIM
from SetExperiment import SetExperiment
from ReadProcessor import ReadProcessor

In [5]:
self = BSReadSim(ref_fasta = ref_fasta, outdir=outdir, prefix=prefix, overwrite_db=True, num_reads=10**5,
                 gzip=False, shuffle=False)

Initiating experiment...
Initiating methylation profile...

[Initiating meth_db] for chr10...
Filling with beta distribution for chr10...
Processed 187408 sites from contig chr10

[Initiating meth_db] for chr11...
Filling with beta distribution for chr11...
Processed 187844 sites from contig chr11

[Initiating meth_db] for chr12...
Filling with beta distribution for chr12...
Processed 184249 sites from contig chr12

[Initiating meth_db] for chr13...
Filling with beta distribution for chr13...
Processed 142075 sites from contig chr13

[Initiating meth_db] for chr14...
Filling with beta distribution for chr14...
Processed 154158 sites from contig chr14

[Initiating meth_db] for chr15...
Filling with beta distribution for chr15...
Processed 2252 sites from contig chr15


../../Simulate/StreamReads.py:42: UserWarning: Fastq file exists, will overwrite... /home/wbguo/iproject/BSReadSim/test/outdir//sim_1.fastq
  warnings.warn(f'Fastq file exists, will overwrite... {fastq_file}')
../../Simulate/StreamReads.py:42: UserWarning: Fastq file exists, will overwrite... /home/wbguo/iproject/BSReadSim/test/outdir//sim_2.fastq
  warnings.warn(f'Fastq file exists, will overwrite... {fastq_file}')


In [6]:
cmd_part = [self.htsim_path, self.ref_fasta] + [str(item) for key_val in self.htsim_opts.items() for item in key_val]

contig_id = 'chr10'

sim_cmd  = cmd_part + ['-c', contig_id] + ['-n', str(self.count_dict[contig_id])]

read_gen = iter(StreamHTSIM(sim_cmd=sim_cmd, pair_end=self.pair_end)) # only output 1 header for -c TODO:
var_contig, sim_data= next(read_gen)                                    # the first element of generator is variants
self.curr_contig= var_contig                                            # update the profiles
self.var_profile= self.meth_set.set_var_meth(var_contig, sim_data)      # a dict, can be empty
pos_map, meth_arr, _  = self.meth_db.load_contig(var_contig)                              # [pos_map, meth_arr, status]
# self.meth_db.load_contig_share(var_contig)                              # [pos_map, meth_arr, status]

Simulating whole genome reads:
Reference genome file: /home/wbguo/iproject/BSReadSim/test/data/ref/BSB_test.fa
[main] Calculating the total length and effective length of the reference sequences...
[main] Contig chr10 specified, contig length: 423500, effective length: 423500
[main] No VCF input, will generate SNP randomly if mutation rate is nonzero
[htsim] seed = 1679077558
[sim_core] contig 'chr10': simulate 21589 reads...


In [7]:
self.processor  = ReadProcessor(meth_arr= meth_arr,
                                pos_map = pos_map,
                                var_profile = self.var_profile,
                                experiment  = self.experiment)

In [8]:
self.data_processor = DataProcessor(read_gen = read_gen, n_workers=8, 
                                    processor=self.processor, fastq_out=self.fastq_out)

In [9]:
import cProfile
cProfile.run('self.data_processor.start_processing_2q()')

[sim_core] Generated 21589 read pairs, with 2355 contain SNP, 500 contain INDEL


         6499382 function calls in 1468.052 seconds

   Ordered by: standard name

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
        1    0.000    0.000 1468.052 1468.052 <string>:1(<module>)
        1    0.084    0.084 1468.052 1468.052 DataProcessor4.py:21(start_processing_2q)
    43178    1.310    0.000   46.815    0.001 StreamHTSIM.py:103(process_read_lines)
    21590    0.035    0.000   47.360    0.002 StreamHTSIM.py:19(__iter__)
    21591    0.147    0.000   47.325    0.002 StreamHTSIM.py:53(collect_reads)
    21590    0.036    0.000    0.362    0.000 StreamHTSIM.py:69(get_line)
    21589    0.095    0.000    0.300    0.000 _base.py:316(__init__)
        1    0.000    0.000    0.000    0.000 _base.py:632(__enter__)
        1    0.000    0.000    0.000    0.000 _base.py:635(__exit__)
    43178    0.048    0.000    0.713    0.000 _methods.py:54(_any)
   129534    0.472    0.000    0.507    0.000 _ufunc_config.py:131(geterr)
   129534    0.499    0.000 

# no queue, n threads output to file with write lock

In [ ]:
import importlib
import BSReadSim_queue
importlib.reload(BSReadSim_queue)
import DataProcessor3
importlib.reload(DataProcessor3)

from BSReadSim_queue import BSReadSim
from DataProcessor3 import DataProcessor
from LockedIterator import LockedIterator
from SetMethylation import SetMethylation
from StreamReads import StreamReads
from StreamHTSIM import StreamHTSIM
from SetExperiment import SetExperiment
from ReadProcessor import ReadProcessor

In [ ]:
self = BSReadSim(ref_fasta = ref_fasta, outdir=outdir, prefix=prefix, overwrite_db=True, num_reads=10**5,
                 gzip=False, shuffle=False)

In [ ]:
cmd_part = [self.htsim_path, self.ref_fasta] + [str(item) for key_val in self.htsim_opts.items() for item in key_val]

contig_id = 'chr10'

sim_cmd  = cmd_part + ['-c', contig_id] + ['-n', str(self.count_dict[contig_id])]

read_gen = iter(StreamHTSIM(sim_cmd=sim_cmd, pair_end=self.pair_end)) # only output 1 header for -c TODO:
var_contig, sim_data= next(read_gen)                                    # the first element of generator is variants
self.curr_contig= var_contig                                            # update the profiles
self.var_profile= self.meth_set.set_var_meth(var_contig, sim_data)      # a dict, can be empty
pos_map, meth_arr, _  = self.meth_db.load_contig(var_contig)                              # [pos_map, meth_arr, status]
# self.meth_db.load_contig_share(var_contig)                              # [pos_map, meth_arr, status]

In [ ]:
self.processor  = ReadProcessor(meth_arr= meth_arr,
                                pos_map = pos_map,
                                var_profile = self.var_profile,
                                experiment  = self.experiment)

In [ ]:
self.data_processor = DataProcessor(read_gen = read_gen, n_workers=8, 
                                    processor=self.processor, fastq_out=self.fastq_out)

In [ ]:
import cProfile
cProfile.run('self.data_processor.start_processing_2q()')

# out_queue only with multiple process

In [ ]:
import importlib
import BSReadSim_queue
importlib.reload(BSReadSim_queue)

from BSReadSim_queue import BSReadSim
from DataProcessor2 import DataProcessor
from LockedIterator import LockedIterator
from SetMethylation import SetMethylation
from StreamReads import StreamReads
from StreamHTSIM import StreamHTSIM
from SetExperiment import SetExperiment
from ReadProcessor import ReadProcessor

In [ ]:
self = BSReadSim(ref_fasta = ref_fasta, outdir=outdir, prefix=prefix, overwrite_db=True, num_reads=10**6,
                 gzip=False, shuffle=False)
cmd_part = [self.htsim_path, self.ref_fasta] + [str(item) for key_val in self.htsim_opts.items() for item in key_val]

contig_id= 'chr10'
sim_cmd  = cmd_part + ['-c', contig_id] + ['-n', str(self.count_dict[contig_id])]

read_gen = LockedIterator(StreamHTSIM(sim_cmd=sim_cmd, pair_end=self.pair_end)) # only output 1 header for -c TODO:
var_contig, sim_data= next(read_gen)                                    # the first element of generator is variants
self.curr_contig= var_contig                                            # update the profiles
self.var_profile= self.meth_set.set_var_meth(var_contig, sim_data)      # a dict, can be empty
self.meth_db.load_contig_share(var_contig)                              # [pos_map, meth_arr, status]

self.processor  = ReadProcessor(meth_arr= self.meth_db.shared_meth_arr,
                                    pos_map = self.meth_db.shared_pos_map,
                                    var_profile = self.var_profile,
                                    experiment  = self.experiment)

In [ ]:
self.data_processor = DataProcessor(read_gen=read_gen, n_process=8, 
                                    processor=self.processor, fastq_out=self.fastq_out)

In [ ]:
self.data_processor.start_processing_mp()

In [ ]:
self.data_processor.stop()

In [ ]:
self.data_processor.out_queue.qsize()

# out_queue only with multiple threads

In [ ]:
import importlib
import BSReadSim_queue
importlib.reload(BSReadSim_queue)

from BSReadSim_queue import BSReadSim
from DataProcessor_thread import DataProcessor
from LockedIterator import LockedIterator
from SetMethylation import SetMethylation
from StreamReads import StreamReads
from StreamHTSIM import StreamHTSIM
from SetExperiment import SetExperiment
from ReadProcessor import ReadProcessor

In [ ]:
self = BSReadSim(ref_fasta = ref_fasta, outdir=outdir, prefix=prefix, overwrite_db=True, num_reads=10**6,
                 gzip=False, shuffle=False)
cmd_part = [self.htsim_path, self.ref_fasta] + [str(item) for key_val in self.htsim_opts.items() for item in key_val]

contig_id= 'chr10'
sim_cmd  = cmd_part + ['-c', contig_id] + ['-n', str(self.count_dict[contig_id])]

read_gen = LockedIterator(StreamHTSIM(sim_cmd=sim_cmd, pair_end=self.pair_end)) # only output 1 header for -c TODO:
var_contig, sim_data= next(read_gen)                                    # the first element of generator is variants
self.curr_contig= var_contig                                            # update the profiles
self.var_profile= self.meth_set.set_var_meth(var_contig, sim_data)      # a dict, can be empty
pos_map, meth_arr, _  = self.meth_db.load_contig(var_contig)                              # [pos_map, meth_arr, status]

self.processor  = ReadProcessor(meth_arr= meth_arr,
                                pos_map = pos_map,
                                var_profile = self.var_profile,
                                experiment  = self.experiment)

In [ ]:
self.data_processor = DataProcessor(read_gen=read_gen, n_workers=8,
                                    processor=self.processor, fastq_out=self.fastq_out)

In [ ]:
self.data_processor.start_processing_mt()

In [ ]:
self.data_processor.out_queue.qsize()

In [ ]:
x = self.data_processor.pool._threads

In [ ]:
x.